# Day 1 · LLM Agent 기초와 안전한 Tool Calling

이 노트북은 **8/23(일) 09:00–18:00** 수업의 따라하기 실습입니다. 오전에는 LLM과 Agent의 차이를 이해하고, 오후에는 결제형 API 없이도 실행되는 안전한 Agent loop를 완성합니다.

> 핵심 원칙: 모델은 결정을 제안하고, 코드는 권한을 검증하며, 외부 쓰기는 사람이 승인합니다.

## 오늘의 완료 조건

- Python과 작업 폴더를 확인한다.
- `model → tool call → validation → execution → observation` 흐름을 설명한다.
- 알 수 없는 도구, 누락 인자, 경로 탈출, 중복 호출을 안전하게 처리한다.
- 샘플 한국어 회의문을 읽고 구조화된 초안을 만든다.
- Ollama가 있으면 로컬 모델 호출까지 확인하고, 없으면 같은 인터페이스로 계속 진행한다.

In [ ]:
from pathlib import Path
import json
import platform
import sys

ROOT = Path.cwd()
if ROOT.name == 'day1':
    ROOT = ROOT.parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Python:', sys.version.split()[0])
print('OS:', platform.platform())
print('Project root:', ROOT)
assert (ROOT / 'src/day1_agent.py').exists(), '저장소 루트에서 실행하거나 ROOT를 수정하세요.'

## 1. 먼저 데이터와 권한의 경계를 본다

Agent에게 운영체제 전체 파일 권한을 주지 않습니다. 오늘의 `read_public_text` 도구는 프로젝트 폴더 안의 `.txt`와 `.md`만 읽도록 설계했습니다. 실제 회사 데이터 대신 비식별 교육용 샘플을 사용합니다.

In [ ]:
meeting_path = ROOT / 'data/meeting_sample_ko.txt'
meeting_text = meeting_path.read_text(encoding='utf-8')
print(meeting_text)

### 관찰 질문

1. 모델이 추측하면 안 되는 정보는 무엇인가요?
2. 담당자와 마감일의 근거 문장은 어디인가요?
3. 자동 발행 전에 사람이 봐야 하는 조건은 무엇인가요?

## 2. Tool schema와 registry 확인

schema는 모델에게 무엇을 요청할 수 있는지 알려주고, registry는 실제 실행 가능한 함수만 연결합니다. 둘이 분리되어야 설명과 권한을 독립적으로 검토할 수 있습니다.

In [ ]:
from src.day1_agent import TOOL_REGISTRY, TOOL_SCHEMAS, SafeToolExecutor

print('허용된 도구:', list(TOOL_REGISTRY))
print(json.dumps({k: {'description': v['description'], 'required': list(v['required'])} for k, v in TOOL_SCHEMAS.items()}, ensure_ascii=False, indent=2))

## 3. 정상 호출과 observation

모델이 바로 함수를 실행하지 않습니다. `SafeToolExecutor`가 이름·인자·경로를 검사한 뒤 결과를 공통 envelope로 감쌉니다.

In [ ]:
executor = SafeToolExecutor(workspace=ROOT)
result = executor.execute('read_public_text', {'path': 'data/meeting_sample_ko.txt'})
print(json.dumps(result.to_dict(), ensure_ascii=False, indent=2)[:1200])
assert result.ok

## 4. 실패를 일부러 만든다

Agent 수업에서 정상 결과만 보면 운영 준비가 되지 않습니다. 아래 네 가지 실패를 고치지 말고 먼저 관찰합니다.

- 등록되지 않은 도구
- 필수 인자 누락
- 워크스페이스 밖으로 나가는 경로
- 존재하지 않는 파일

In [ ]:
failure_cases = [
    ('delete_everything', {}),
    ('read_public_text', {}),
    ('read_public_text', {'path': '../../etc/passwd'}),
    ('read_public_text', {'path': 'data/not_found.txt'}),
]

for name, arguments in failure_cases:
    failed = executor.execute(name, arguments)
    print(name, '→', failed.error_code, '|', failed.message)
    assert failed.ok is False

### 예외 정책 표

| 오류 | 자동 재시도 | 휴먼 검토 | 권장 처리 |
|---|---:|---:|---|
| schema/인자 오류 | 1회 재계획 | 반복 시 | 모델에게 validation message 반환 |
| 일시적 timeout/429 | 지수 backoff 2–3회 | 계속 실패 시 | `retry_after` 기록 |
| 파일 없음 | 하지 않음 | 필요 | 사용자가 입력을 다시 선택 |
| 정책 위반/권한 부족 | 하지 않음 | 반드시 | 차단 사실과 근거 기록 |
| 외부 쓰기 | 하지 않음 | 실행 전 필수 | 승인·수정·거절 interrupt |

## 5. 중복 실행 방지

Tool call을 hash로 식별해 같은 요청은 cache에서 반환합니다. 읽기에서는 단순 최적화지만, Day 4의 GitHub comment에서는 이 패턴이 중복 게시를 막는 idempotency key가 됩니다.

In [ ]:
first = executor.execute('read_public_text', {'path': 'data/meeting_sample_ko.txt'})
second = executor.execute('read_public_text', {'path': 'data/meeting_sample_ko.txt'})
print('첫 호출 cached:', first.cached)
print('두 번째 호출 cached:', second.cached)
assert second.cached is True

## 6. 결정론적 planner로 Agent loop 완성

첫날에는 LLM 대신 규칙 기반 planner를 사용합니다. 다음 날 planner만 로컬 LLM으로 바꾸면 executor와 테스트를 그대로 재사용할 수 있습니다. 이것이 provider를 harness 뒤로 숨기는 이유입니다.

In [ ]:
from src.day1_agent import run_agent_once

event = run_agent_once('data/meeting_sample_ko.txt를 읽어줘', workspace=ROOT)
print(json.dumps(event, ensure_ascii=False, indent=2)[:1600])
assert event['tool_result']['ok'] is True
assert event['needs_human_review'] is False

## 7. 회의 결과 fixture와 Human Approval

아직 실제 LLM이 없어도 downstream contract를 먼저 만들 수 있습니다. 아래 결과는 Day 3의 STT와 Day 5의 LangGraph가 지켜야 할 출력 예시입니다. **승인 플래그를 통과하기 전에는 GitHub나 외부 문서에 쓰지 않습니다.**

In [ ]:
from src.day1_agent import build_day1_summary

draft = build_day1_summary(meeting_text)
print(json.dumps(draft, ensure_ascii=False, indent=2))
assert draft['requires_human_approval'] is True
assert all(item.get('owner') and item.get('due_date') for item in draft['action_items'])

### 2인 1조 Human Review (7분)

1. 원문 근거와 초안을 나란히 봅니다.
2. 담당자·기한·범위가 원문에 있는지 표시합니다.
3. `승인 / 수정 후 승인 / 거절` 중 하나를 선택합니다.
4. 거절이면 재실행보다 먼저 거절 사유를 구조화합니다.
5. Day 5에서는 이 순간을 LangGraph `interrupt()`와 LangSmith feedback으로 남깁니다.

## 8. 선택 실습: 로컬 Ollama 호출

Ollama를 설치하고 모델을 받은 학습자만 실행합니다. 서버가 없더라도 셀은 예외를 공통 오류 구조로 반환하므로 노트북 전체가 중단되지 않습니다.

```bash
ollama pull qwen3:4b
ollama run qwen3:4b
```

In [ ]:
from src.day1_agent import call_ollama

ollama_result = call_ollama(
    '다음 문장을 20자 이내로 요약하세요: 외부 발행 전에는 반드시 사람이 승인한다.',
    timeout=3,
)
if ollama_result['ok']:
    print(ollama_result['data'].get('response', ''))
else:
    print('[대체 경로]', ollama_result['error_code'], '-', ollama_result['message'])

## 9. 테스트를 실행한다

터미널에서 아래 명령을 실행합니다. 테스트 통과 화면을 캡처해 실습 기록에 붙입니다.

```bash
python -m pytest -q
```

테스트가 없다면 Agent가 그럴듯한 결과를 냈다는 사실만 알 수 있습니다. 테스트가 있으면 허용 도구, 입력 계약, 경로 정책, 중복 방지를 반복 검증할 수 있습니다.

## 10. Git checkpoint

```bash
git status
git add src/day1_agent.py tests/test_day1_agent.py data/meeting_sample_ko.txt materials/day1/01_agent_foundation.ipynb
git commit -m "feat: add safe day1 tool loop"
git log --oneline -5
```

완료 기준은 commit 자체가 아니라 **diff를 설명하고 검증 명령을 재실행할 수 있는 상태**입니다.

## Exit ticket

다음 세 문장을 본인의 말로 완성합니다.

1. LLM과 Agent의 차이는 ______이다.
2. Tool Calling에서 가장 먼저 검증할 것은 ______이다.
3. 외부 쓰기 전에 Human Approval이 필요한 이유는 ______이다.

**다음 날 예고:** schema를 JSON/Pydantic으로 강화하고, 실제 로컬 LLM과 LangChain tool binding을 연결합니다.